In [ ]:
import pandas as pd
import os

# Updated folder path (your new path)
path = r"D:\Telangana PDS Analytics\data\Transaction_data_2023_2025"

# 🔍 Check folder exists
if not os.path.exists(path):
    raise Exception(f"Folder path does not exist: {path}")

# 📄 Get all CSV files
files = [f for f in os.listdir(path) if f.endswith(".csv")]

if len(files) == 0:
    raise Exception("No CSV files found in folder!")

df_list = []

# 🔄 Read all CSV files safely
for file in files:
    full_path = os.path.join(path, file)
    print("Reading:", full_path)

    try:
        temp_df = pd.read_csv(full_path, low_memory=False, encoding="utf-8")
        df_list.append(temp_df)

    except UnicodeDecodeError:
        temp_df = pd.read_csv(full_path, low_memory=False, encoding="latin1")
        df_list.append(temp_df)

    except Exception as e:
        print(f"❌ Error reading {file}: {e}")

# 🚨 Check if data loaded
if len(df_list) == 0:
    raise Exception("No data loaded from CSV files!")

# 🔗 Merge all CSVs
df = pd.concat(df_list, ignore_index=True)

# 📊 Final shape
print("✅ Final Shape:", df.shape)

# 💾 Save output file
output_path = os.path.join(path, "Transaction_data_combined.csv")
df.to_csv(output_path, index=False)

print("✅ File saved at:", output_path)

Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_10_2023.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_11_2023.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_11_2024.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_12_2023.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_12_2024.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_1_2023.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_1_2024.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_1_2025.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\shop-wise-trans-details_2_2023.csv
Reading: D:\Telangana PDS Analytics\data\Transaction_data_2023_2025\

In [33]:
print("Total files combined:", len(files))

Total files combined: 28


In [34]:
df.duplicated().sum()

np.int64(0)

In [35]:
df.shape

(482407, 19)

In [37]:
num_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

print("Numerical Columns:", len(num_cols))

print(num_cols)

Numerical Columns: 17
Index(['distCode', 'officeCode', 'shopNo', 'month', 'year', 'noOfRcs',
       'noOfTrans', 'riceAfsc', 'riceFsc', 'riceAap', 'wheat', 'sugar',
       'rgdal', 'kerosene', 'totalAmount', 'salt', 'otherShopTransCnt'],
      dtype='object')


In [38]:
print(df['otherShopTransCnt'].describe())

count    482407.000000
mean        158.526632
std         237.231484
min           0.000000
25%          18.000000
50%          56.000000
75%         198.000000
max        5070.000000
Name: otherShopTransCnt, dtype: float64


In [39]:
cat_cols = df.select_dtypes(
    include=["object"]
).columns

print("Categorical Columns:", len(cat_cols))

print(cat_cols)

Categorical Columns: 2
Index(['distName', 'officeName'], dtype='object')


In [40]:
#check Zero or null value in data
summary = pd.DataFrame({
    "Column": df.columns,
    "Null_Count": df.isnull().sum().values,
    "Zero_Count": [
        (df[col] == 0).sum()
        if pd.api.types.is_numeric_dtype(df[col])
        else "-"
        for col in df.columns
    ]
})
print(summary)


               Column  Null_Count Zero_Count
0            distCode           0          0
1            distName           0          -
2          officeCode           0          0
3          officeName           0          -
4              shopNo           0          0
5               month           0          0
6                year           0          0
7             noOfRcs           0          0
8           noOfTrans           0          0
9            riceAfsc           0       2352
10            riceFsc           0        155
11            riceAap           0     447174
12              wheat           0     445613
13              sugar           0     424961
14              rgdal           0     482407
15           kerosene           0     480713
16        totalAmount           0     415146
17               salt           0     481983
18  otherShopTransCnt           0       3101


In [41]:
# Check zero percentage
zero_percent = (df == 0).mean() * 100

In [42]:

print(zero_percent.sort_values(ascending=False))

rgdal                100.000000
salt                  99.912107
kerosene              99.648844
riceAap               92.696416
wheat                 92.372830
sugar                 88.091798
totalAmount           86.057209
otherShopTransCnt      0.642818
riceAfsc               0.487555
riceFsc                0.032131
distCode               0.000000
officeCode             0.000000
distName               0.000000
noOfRcs                0.000000
noOfTrans              0.000000
shopNo                 0.000000
officeName             0.000000
month                  0.000000
year                   0.000000
dtype: float64


In [43]:
df.shape


(482407, 19)

In [44]:
cols_to_drop = ["rgdal", "salt", "kerosene"]

df.drop(columns=cols_to_drop, inplace=True)

print("Dropped columns:", cols_to_drop)

Dropped columns: ['rgdal', 'salt', 'kerosene']


In [45]:
important_cols = [
    "riceAap",
    "wheat",
    "sugar",
    "totalAmount"
]

In [46]:
for col in important_cols:
    df[col + "_available"] = df[col].apply(lambda x: 1 if x > 0 else 0)

In [47]:
zero_percent_after = (df == 0).mean() * 100

print(zero_percent_after.sort_values(ascending=False))

riceAap_available        92.696416
riceAap                  92.696416
wheat_available          92.372830
wheat                    92.372830
sugar                    88.091798
sugar_available          88.091798
totalAmount              86.057209
totalAmount_available    86.057209
otherShopTransCnt         0.642818
riceAfsc                  0.487555
riceFsc                   0.032131
distCode                  0.000000
officeName                0.000000
officeCode                0.000000
distName                  0.000000
noOfTrans                 0.000000
shopNo                    0.000000
month                     0.000000
year                      0.000000
noOfRcs                   0.000000
dtype: float64


In [18]:
#Need not to be Drop these coloumn
drop_cols = [
    "riceAap",
    "riceAap_available",
    "wheat",
    "wheat_available"
]

df.drop(columns=drop_cols, inplace=True)

print("Dropped Columns:")
print(drop_cols)

Dropped Columns:
['riceAap', 'riceAap_available', 'wheat', 'wheat_available']


In [49]:
print("\nRemaining Columns:")
print(df.columns)

print("\nDataset Shape:")
print(df.shape)



Remaining Columns:
Index(['distCode', 'distName', 'officeCode', 'officeName', 'shopNo', 'month',
       'year', 'noOfRcs', 'noOfTrans', 'riceAfsc', 'riceFsc', 'riceAap',
       'wheat', 'sugar', 'totalAmount', 'otherShopTransCnt',
       'riceAap_available', 'wheat_available', 'sugar_available',
       'totalAmount_available'],
      dtype='object')

Dataset Shape:
(482407, 20)


In [50]:
summary = pd.DataFrame({
    "Column": df.columns,
    "Null_Count": df.isnull().sum().values,
    "Zero_Count": [
        (df[col] == 0).sum()
        if pd.api.types.is_numeric_dtype(df[col])
        else "-"
        for col in df.columns
    ]
})
print(summary)

                   Column  Null_Count Zero_Count
0                distCode           0          0
1                distName           0          -
2              officeCode           0          0
3              officeName           0          -
4                  shopNo           0          0
5                   month           0          0
6                    year           0          0
7                 noOfRcs           0          0
8               noOfTrans           0          0
9                riceAfsc           0       2352
10                riceFsc           0        155
11                riceAap           0     447174
12                  wheat           0     445613
13                  sugar           0     424961
14            totalAmount           0     415146
15      otherShopTransCnt           0       3101
16      riceAap_available           0     447174
17        wheat_available           0     445613
18        sugar_available           0     424961
19  totalAmount_avai

In [51]:
keep_cols = [
    "riceAfsc",
    "riceFsc",
    "otherShopTransCnt",
    "sugar",
    "totalAmount"
]

In [52]:
df.drop(
    columns=[
        "sugar_available",
        "totalAmount_available"
    ],
    inplace=True
)

In [53]:
analysis_df = df.copy()

analysis_df["sugar"] = analysis_df["sugar"].replace(0, pd.NA)

analysis_df["totalAmount"] = analysis_df["totalAmount"].replace(0, pd.NA)

In [54]:
analysis_df = df.copy()

In [55]:
analysis_df["sugar"] = analysis_df["sugar"].replace(0, pd.NA)

In [56]:
analysis_df["sugar"].mean()

np.float64(29.76811266232636)

In [57]:
print(analysis_df[["sugar", "totalAmount"]].head(10))

  sugar  totalAmount
0  <NA>          0.0
1  <NA>          0.0
2  <NA>          0.0
3  <NA>          0.0
4  <NA>          0.0
5  <NA>          0.0
6  <NA>          0.0
7  <NA>          0.0
8  <NA>          0.0
9  <NA>          0.0


In [58]:
print("Final Shape:", df.shape)
print("Any Nulls:", df.isnull().sum().sum())
print("Any Duplicates:", df.duplicated().sum())

Final Shape: (482407, 18)
Any Nulls: 0
Any Duplicates: 0


In [60]:
df.to_csv("../data/clean_transaction_data.csv", index=False)